First import modules and initialize Earth Engine.

In [ ]:

# standard modules
import io
import json
import os
from pathlib import Path
import time

# specialized modules
import ee
import geemap
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm

# initialize the Earth Engine module.
ee.Initialize(project='trinity-438000')

Open the AOI vector file and check it on a map.

In [ ]:
# read AOIs
# This works whether the notebook is launched from the repo root or from assets/.
this_dir = Path.cwd()
repo_dir = this_dir if (this_dir / 'assets').exists() else this_dir.parent

def first_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'None of these paths exist: {candidates}')

# Broad analysis/export AOI.
aoi_path = first_existing_path([
    this_dir / 'rio_san_juan_aoi.geojson',
    repo_dir / 'assets' / 'rio_san_juan_aoi.geojson',
])

# Smaller comparison/reference AOI inside the Rio San Juan box.
rio_aoi_path = first_existing_path([
    this_dir / 'rio_indio_aoi.geojson',
    repo_dir / 'untracked_qgis' / 'susques' / 'rio_indio_aoi.geojson',
])

aoi_path, rio_aoi_path

In [ ]:
aoi = gpd.read_file(aoi_path).to_crs(4326)
rio_aoi = gpd.read_file(rio_aoi_path).to_crs(4326)

gee_json = json.loads(aoi[['geometry']].to_json())
gee_aoi = geemap.geojson_to_ee(gee_json)

gee_rio_json = json.loads(rio_aoi[['geometry']].to_json())
gee_rio_aoi = geemap.geojson_to_ee(gee_rio_json)

# inspect the GeoJSON as an EEObject through geemap.
test_map = geemap.Map(basemap='SATELLITE')
test_map.centerObject(gee_aoi, 9)

# Style the AOIs explicitly so both are visible on the map.
aoi_style = {'color': 'FF0000', 'fillColor': '00000000', 'width': 3}
rio_aoi_style = {'color': 'FFFF00', 'fillColor': '00000000', 'width': 3}
test_map.addLayer(gee_aoi.style(**aoi_style), {}, 'Broad Rio San Juan AOI')
test_map.addLayer(gee_rio_aoi.style(**rio_aoi_style), {}, 'Rio Indio AOI')

test_map


Get the vertices of the broad AOI and Rio Indio AOI to use in Earth Engine filters.


In [ ]:
# get extents for both AOIs
def bounds_to_verts(gdf):
    minx, miny, maxx, maxy = gdf.total_bounds
    return [
        [float(minx), float(miny)],
        [float(minx), float(maxy)],
        [float(maxx), float(maxy)],
        [float(maxx), float(miny)],
        [float(minx), float(miny)],
    ]

# Broad box from the original Rio San Juan AOI.
verts = bounds_to_verts(aoi)

# Smaller AOI used for Rio Indio wet-season composites.
rio_verts = bounds_to_verts(rio_aoi)

verts, rio_verts


Wet-season composite periods

Use these periods to download annual wet-season composites for the Rio Indio AOI. Students can calculate turbidity indices later in QGIS from the exported bands.


In [ ]:
# Wet season for the Caribbean side of Nicaragua.
# End dates are exclusive in Earth Engine, so November composites end on December 1.
WET_START_MONTH = 5
WET_END_MONTH = 11

composite_periods = {
    'landsat_5_7': {
        'start_year': 2000,
        'end_year': 2011,
        'start_month': WET_START_MONTH,
        'end_month': WET_END_MONTH,
        'scale': 30,
        'short_name': 'l57',
    },
    'landsat_8_9': {
        'start_year': 2013,
        'end_year': 2025,
        'start_month': WET_START_MONTH,
        'end_month': WET_END_MONTH,
        'scale': 30,
        'short_name': 'l89',
    },
    'sentinel_2': {
        'start_year': 2018,
        'end_year': 2025,
        'start_month': WET_START_MONTH,
        'end_month': WET_END_MONTH,
        'scale': 10,
        'short_name': 's2',
    },
}

# Use this region for Rio Indio downloads.
rio_export_region = gee_rio_aoi.geometry()


def wet_season_date_range(year):
    start = ee.Date.fromYMD(year, WET_START_MONTH, 1)
    end = ee.Date.fromYMD(year, WET_END_MONTH, 1).advance(1, 'month')
    return start, end


# Quick check: these are the image years that will be requested for each sensor group.
for sensor, period in composite_periods.items():
    years = list(range(period['start_year'], period['end_year'] + 1))
    print(sensor, years[0], 'to', years[-1], f"({len(years)} composites)")
